In [1]:
import os
import cv2

# Paths
VIDEO_DATASET_DIR = r"G:\ASD Without Image\ASD Vedio\ASD_Videos"
OUTPUT_DIR = r"G:\ASD Without Image\ASD Vedio"

FRAME_INTERVAL = 30  # extract 1 frame every 30 frames

# Create output structure
os.makedirs(OUTPUT_DIR, exist_ok=True)

for class_name in os.listdir(VIDEO_DATASET_DIR):
    class_dir = os.path.join(VIDEO_DATASET_DIR, class_name)
    if not os.path.isdir(class_dir):
        continue
    
    out_class_dir = os.path.join(OUTPUT_DIR, class_name)
    os.makedirs(out_class_dir, exist_ok=True)

    for vid_name in os.listdir(class_dir):
        if not vid_name.lower().endswith(".avi"):
            continue
        
        vid_path = os.path.join(class_dir, vid_name)
        cap = cv2.VideoCapture(vid_path)
        
        frame_count, saved_count = 0, 0
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            if frame_count % FRAME_INTERVAL == 0:
                out_path = os.path.join(out_class_dir, f"{os.path.splitext(vid_name)[0]}_frame{frame_count}.jpg")
                cv2.imwrite(out_path, frame)
                saved_count += 1
            frame_count += 1
        cap.release()
        print(f"Extracted {saved_count} frames from {vid_name} → {out_class_dir}")


Extracted 19 frames from autism1.avi → G:\ASD Without Image\ASD Vedio\ASD
Extracted 17 frames from autism2.avi → G:\ASD Without Image\ASD Vedio\ASD
Extracted 23 frames from autism3.avi → G:\ASD Without Image\ASD Vedio\ASD
Extracted 15 frames from autism4.avi → G:\ASD Without Image\ASD Vedio\ASD
Extracted 13 frames from autism5.avi → G:\ASD Without Image\ASD Vedio\ASD
Extracted 12 frames from autism6.avi → G:\ASD Without Image\ASD Vedio\ASD
Extracted 10 frames from autism7.avi → G:\ASD Without Image\ASD Vedio\ASD
Extracted 18 frames from autism8.avi → G:\ASD Without Image\ASD Vedio\ASD
Extracted 5 frames from autism9.avi → G:\ASD Without Image\ASD Vedio\ASD
Extracted 5 frames from non_autism1.avi → G:\ASD Without Image\ASD Vedio\NON_ASD
Extracted 18 frames from non_autism10.avi → G:\ASD Without Image\ASD Vedio\NON_ASD
Extracted 15 frames from non_autism4.avi → G:\ASD Without Image\ASD Vedio\NON_ASD
Extracted 28 frames from non_autism5.avi → G:\ASD Without Image\ASD Vedio\NON_ASD
Extract

In [4]:
import os
import cv2
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split

# ======================
# 1. Parameters
# ======================
# Folder structure expected:
# VIDEO_DATASET_DIR/
#     class1/
#         video1.avi
#         video2.avi
#     class2/
#         video3.avi
#         ...

VIDEO_DATASET_DIR = r"G:\ASD Without Image\ASD Vedio\ASD_Videos"  # your AVI dataset (per-class subfolders)
FRAME_DATASET_DIR = r"G:\ASD Without Image\ASD Vedio\frames"      # extracted frames root (per-class subfolders)

FRAME_INTERVAL = 30   # save every 30th frame
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 10
MODEL_PATH = "autism_classifier_best.pth"
CLASS_MAP_PATH = "class_mapping.json"

# ======================
# 2. Step 1 – Extract Frames
# ======================
os.makedirs(FRAME_DATASET_DIR, exist_ok=True)

print("🎥 Extracting frames from videos...")

if not os.path.isdir(VIDEO_DATASET_DIR):
    raise FileNotFoundError(f"VIDEO_DATASET_DIR does not exist: {VIDEO_DATASET_DIR}")

class_dirs = [d for d in os.listdir(VIDEO_DATASET_DIR)
              if os.path.isdir(os.path.join(VIDEO_DATASET_DIR, d))]

if not class_dirs:
    raise RuntimeError(
        f"No class subfolders found in {VIDEO_DATASET_DIR}.\n"
        f"Expected structure: {VIDEO_DATASET_DIR}\\class_name\\video.avi"
    )

for class_name in class_dirs:
    class_dir = os.path.join(VIDEO_DATASET_DIR, class_name)
    out_class_dir = os.path.join(FRAME_DATASET_DIR, class_name)
    os.makedirs(out_class_dir, exist_ok=True)

    print(f"\n📂 Processing class: {class_name}")
    video_files = [v for v in os.listdir(class_dir) if v.lower().endswith(".avi")]

    if not video_files:
        print(f"  ⚠️ No .avi videos found in {class_dir}. Skipping this class.")
        continue

    for vid_name in video_files:
        vid_path = os.path.join(class_dir, vid_name)
        cap = cv2.VideoCapture(vid_path)

        if not cap.isOpened():
            print(f"  ⚠️ Could not open video: {vid_path}")
            continue

        frame_count, saved_count = 0, 0
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            if frame_count % FRAME_INTERVAL == 0:
                frame_filename = f"{os.path.splitext(vid_name)[0]}_frame{frame_count}.jpg"
                out_path = os.path.join(out_class_dir, frame_filename)
                cv2.imwrite(out_path, frame)
                saved_count += 1

            frame_count += 1

        cap.release()
        print(f"  ✅ Extracted {saved_count} frames from {vid_name} → {out_class_dir}")

print("\n✅ Frame extraction complete.")

# Quick sanity check: count images in FRAME_DATASET_DIR
def count_images(root_dir):
    exts = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp")
    count = 0
    for root, _, files in os.walk(root_dir):
        for f in files:
            if f.lower().endswith(exts):
                count += 1
    return count

total_images = count_images(FRAME_DATASET_DIR)
print(f"🖼️ Total images found in {FRAME_DATASET_DIR}: {total_images}")
if total_images == 0:
    raise RuntimeError(
        f"No image files were found in {FRAME_DATASET_DIR}.\n"
        f"Check that frames were actually saved and that the path is correct."
    )

# ======================
# 3. Step 2 – Transforms
# ======================
train_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.2),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# ======================
# 4. Step 3 – Dataset & Split
# ======================
print("\n📚 Building ImageFolder dataset from frames...")

full_dataset = datasets.ImageFolder(FRAME_DATASET_DIR, transform=train_transform)

if len(full_dataset) == 0:
    raise RuntimeError(
        f"ImageFolder found 0 images in {FRAME_DATASET_DIR}.\n"
        f"Ensure structure is:\n"
        f"{FRAME_DATASET_DIR}\\class1\\img1.jpg\n"
        f"{FRAME_DATASET_DIR}\\class2\\img2.jpg\n"
    )

class_names = full_dataset.classes
print("Classes:", class_names)

# Save mapping
with open(CLASS_MAP_PATH, "w") as f:
    json.dump(full_dataset.class_to_idx, f)
print("✅ Saved class mapping to", CLASS_MAP_PATH, ":", full_dataset.class_to_idx)

# Split (80/20)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# Use validation transforms for the val_dataset
val_dataset.dataset.transform = val_transform

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"🧾 Dataset sizes → Train: {len(train_dataset)}, Val: {len(val_dataset)}")

# ======================
# 5. Step 4 – Model
# ======================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n🖥️ Using device: {device}")

# If your torchvision version supports this string API, it's fine.
# If not, replace with: models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model = models.resnet18(weights="IMAGENET1K_V1")

# Fine-tune last block + classifier
for name, param in model.named_parameters():
    if "layer4" in name or "fc" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

# Replace final fully connected layer
model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(model.fc.in_features, len(class_names))
)

model = model.to(device)

# ======================
# 6. Step 5 – Loss & Optimizer
# ======================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

# ======================
# 7. Step 6 – Training Loop
# ======================
print("\n🚀 Starting training...")
best_val_acc = 0.0

for epoch in range(EPOCHS):
    # ---------- Train ----------
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    train_loss = running_loss / max(len(train_loader), 1)
    train_acc = 100.0 * correct / max(total, 1)

    # ---------- Validate ----------
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()

    val_loss = val_loss / max(len(val_loader), 1)
    val_acc = 100.0 * val_correct / max(val_total, 1)

    print(f"Epoch [{epoch + 1}/{EPOCHS}] "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% | "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), MODEL_PATH)
        print(f"  ✅ Saved new best model → {MODEL_PATH} (Val Acc: {val_acc:.2f}%)")

print(f"\n🎯 Best Validation Accuracy: {best_val_acc:.2f}%")
print("🏁 Training complete.")


🎥 Extracting frames from videos...

📂 Processing class: ASD
  ✅ Extracted 19 frames from autism1.avi → G:\ASD Without Image\ASD Vedio\frames\ASD
  ✅ Extracted 17 frames from autism2.avi → G:\ASD Without Image\ASD Vedio\frames\ASD
  ✅ Extracted 23 frames from autism3.avi → G:\ASD Without Image\ASD Vedio\frames\ASD
  ✅ Extracted 15 frames from autism4.avi → G:\ASD Without Image\ASD Vedio\frames\ASD
  ✅ Extracted 13 frames from autism5.avi → G:\ASD Without Image\ASD Vedio\frames\ASD
  ✅ Extracted 12 frames from autism6.avi → G:\ASD Without Image\ASD Vedio\frames\ASD
  ✅ Extracted 10 frames from autism7.avi → G:\ASD Without Image\ASD Vedio\frames\ASD
  ✅ Extracted 18 frames from autism8.avi → G:\ASD Without Image\ASD Vedio\frames\ASD
  ✅ Extracted 5 frames from autism9.avi → G:\ASD Without Image\ASD Vedio\frames\ASD

📂 Processing class: NON_ASD
  ✅ Extracted 5 frames from non_autism1.avi → G:\ASD Without Image\ASD Vedio\frames\NON_ASD
  ✅ Extracted 18 frames from non_autism10.avi → G:\ASD 

In [6]:
import os
import cv2
import json
import torch
import torch.nn as nn
from torchvision import transforms, models
from collections import Counter
from PIL import Image

# ======================
# 1. Parameters
# ======================
MODEL_PATH = "autism_classifier_best.pth"
CLASS_MAP_PATH = "class_mapping.json"
VIDEO_PATH = r"G:\ASD Without Image\ASD Vedio\ASD_Videos\ASD\autism7.avi"   # change this
FRAME_INTERVAL = 30      # process every 30th frame
FRAME_SIZE = (224, 224)  # resize for model

# ======================
# 2. Load Class Mapping
# ======================
with open(CLASS_MAP_PATH) as f:
    class_to_idx = json.load(f)
idx_to_class = {v: k for k, v in class_to_idx.items()}
CLASS_NAMES = [idx_to_class[i] for i in range(len(idx_to_class))]
print("✅ Loaded classes:", CLASS_NAMES)

# ======================
# 3. Preprocessing
# ======================
transform = transforms.Compose([
    transforms.Resize(FRAME_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# ======================
# 4. Load Model
# ======================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.resnet18(weights="IMAGENET1K_V1")
model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(model.fc.in_features, len(CLASS_NAMES))
)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model = model.to(device)
model.eval()

# ======================
# 5. Video Prediction
# ======================
cap = cv2.VideoCapture(VIDEO_PATH)
frame_count, preds = 0, []

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    if frame_count % FRAME_INTERVAL == 0:
        img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        img = Image.fromarray(img)
        img = transform(img).unsqueeze(0).to(device)

        with torch.no_grad():
            outputs = model(img)
            _, predicted = torch.max(outputs, 1)
            preds.append(predicted.item())

    frame_count += 1

cap.release()

# ======================
# 6. Aggregate Results
# ======================
if preds:
    final_pred = Counter(preds).most_common(1)[0][0]
    print(f"🎥 Final Prediction for video: {CLASS_NAMES[final_pred]}")
    print(f"🖼️ Frame-level predictions: {Counter(preds)}")
else:
    print("⚠️ No frames processed from video.")


✅ Loaded classes: ['ASD', 'NON_ASD']
🎥 Final Prediction for video: ASD
🖼️ Frame-level predictions: Counter({0: 10})
